# Token-Reducer Playground

A hands-on notebook to see **every piece of the input/output** the rig touches —
*without* needing a model running for the most important parts.

The flow this rig sits in:

```
mini-swe-agent --(OpenAI request)--> proxy.py --(transform)--> Ollama --(SSE)--> back
                                        |
                                  meter -> rig_calls.jsonl
```

Sections:
1. Anatomy of an OpenAI chat request (the **input**)
2. What each transform does to it (the rig's whole job)
3. Metering — how tokens/cost are accounted (the **output** signal)
4. Live round-trip (optional — needs the proxy or Ollama up)
5. Capture & dissect **real** agent traffic
6. Read the logs

> This notebook imports the rig's *own* `transforms.py` / `meter.py` / `rig_data.py`,
> so what you see here is exactly what the proxy does — no reimplementation, no drift.

In [1]:
import sys, os, json, copy
# Run this notebook from the repo root so the rig modules import cleanly.
sys.path.insert(0, os.getcwd())
import transforms, meter, rig_data

print("cwd               :", os.getcwd())
print("transforms        :", list(transforms._REGISTRY))
print("active (env)      :", transforms.transform_name())

cwd               : /Users/paulgailey/git/token-reducer
transforms        : ['identity', 'dedup', 'prune']
active (env)      : identity


## 1. Anatomy of an OpenAI chat request

This is the **input** every transform receives: a dict in OpenAI Chat Completions
shape. The interesting fields:

- `messages[]` — the conversation. Each has a `role`:
  - `system` — instructions (transforms must never touch this)
  - `user` / `assistant` — plain text turns
  - `assistant` with `tool_calls` — the model asking to run a tool (no text content)
  - `tool` with `tool_call_id` — the **result** of that tool call (often huge: command output)
- `stream` — whether the response comes back as SSE chunks
- the **last** message is the live one the model must answer — transforms must never drop it

`make_convo` below synthesizes a realistic agent loop so both transforms have
something to bite on. **Tweak `n_turns` / `output_size` / `dup` and re-run the
whole notebook** to see the effects change.

In [2]:
def make_convo(n_turns=6, output_size=80, dup=True):
    msgs = [
        {"role": "system", "content": "You are a coding agent. Work step by step."},
        {"role": "user",   "content": "Fix the failing test in app.py"},
    ]
    if dup:  # an EXACT duplicate user message -> dedup will drop it
        msgs.append({"role": "user", "content": "Fix the failing test in app.py"})
    for k in range(n_turns):
        msgs.append({"role": "assistant", "content": None,
            "tool_calls": [{"id": f"call_{k}", "type": "function",
                "function": {"name": "bash",
                             "arguments": json.dumps({"cmd": f"step {k}"})}}]})
        # a big tool output -> prune will truncate the OLD ones
        msgs.append({"role": "tool", "tool_call_id": f"call_{k}",
            "content": f"output of step {k}\n" + ("log line ....\n" * output_size)})
    msgs.append({"role": "user", "content": "What's the fix?"})  # final = live message
    return {"model": "llama3.1", "stream": True, "messages": msgs}

SAMPLE = make_convo()
print(f"{len(SAMPLE['messages'])} messages")
print(json.dumps(SAMPLE['messages'][:5], indent=2))   # peek at the head

16 messages
[
  {
    "role": "system",
    "content": "You are a coding agent. Work step by step."
  },
  {
    "role": "user",
    "content": "Fix the failing test in app.py"
  },
  {
    "role": "user",
    "content": "Fix the failing test in app.py"
  },
  {
    "role": "assistant",
    "content": null,
    "tool_calls": [
      {
        "id": "call_0",
        "type": "function",
        "function": {
          "name": "bash",
          "arguments": "{\"cmd\": \"step 0\"}"
        }
      }
    ]
  },
  {
    "role": "tool",
    "tool_call_id": "call_0",
    "content": "output of step 0\nlog line ....\nlog line ....\nlog line ....\nlog line ....\nlog line ....\nlog line ....\nlog line ....\nlog line ....\nlog line ....\nlog line ....\nlog line ....\nlog line ....\nlog line ....\nlog line ....\nlog line ....\nlog line ....\nlog line ....\nlog line ....\nlog line ....\nlog line ....\nlog line ....\nlog line ....\nlog line ....\nlog line ....\nlog line ....\nlog line ....\nlog line 

## 2. What each transform does

A transform is just a function `request -> request`. **Critically, they mutate the
dict in place** (e.g. `prune` edits `messages[i]["content"]`), so `proxy.py`
always deep-copies the request before applying one. `apply_transform` below does
the same — never hand a transform your only copy.

`show()` prints a per-message map: index, role, char count, and whether the
message is *tool-paired* (an assistant `tool_calls` or a `tool` result — these are
load-bearing and `dedup` refuses to touch them).

In [3]:
def _summary(m):
    role   = m.get("role")
    c      = m.get("content")
    paired = bool(m.get("tool_calls")) or "tool_call_id" in m or role == "tool"
    chars  = len(c) if isinstance(c, str) else 0
    return role, chars, paired

def show(req, label):
    msgs  = req.get("messages", [])
    total = sum(_summary(m)[1] for m in msgs)
    # ~chars/4 is a rough token proxy ONLY; real counts come from the model's usage.
    print(f"\n=== {label} ===  {len(msgs)} msgs, ~{total} chars (~{total // 4} tok)")
    for i, m in enumerate(msgs):
        role, chars, paired = _summary(m)
        print(f"  [{i:>2}] {role:<9} {chars:>6} chars  {'tool-paired' if paired else ''}")

def apply_transform(req, name):
    # deep-copy first, exactly like proxy.py — transforms mutate in place
    return transforms._REGISTRY[name](copy.deepcopy(req))

show(SAMPLE, "ORIGINAL")
for name in ("identity", "dedup", "prune"):
    show(apply_transform(SAMPLE, name), name.upper())


=== ORIGINAL ===  16 msgs, ~6939 chars (~1734 tok)
  [ 0] system        42 chars  
  [ 1] user          30 chars  
  [ 2] user          30 chars  
  [ 3] assistant      0 chars  tool-paired
  [ 4] tool        1137 chars  tool-paired
  [ 5] assistant      0 chars  tool-paired
  [ 6] tool        1137 chars  tool-paired
  [ 7] assistant      0 chars  tool-paired
  [ 8] tool        1137 chars  tool-paired
  [ 9] assistant      0 chars  tool-paired
  [10] tool        1137 chars  tool-paired
  [11] assistant      0 chars  tool-paired
  [12] tool        1137 chars  tool-paired
  [13] assistant      0 chars  tool-paired
  [14] tool        1137 chars  tool-paired
  [15] user          15 chars  

=== IDENTITY ===  16 msgs, ~6939 chars (~1734 tok)
  [ 0] system        42 chars  
  [ 1] user          30 chars  
  [ 2] user          30 chars  
  [ 3] assistant      0 chars  tool-paired
  [ 4] tool        1137 chars  tool-paired
  [ 5] assistant      0 chars  tool-paired
  [ 6] tool        1137 cha

**What you should see:**
- `identity` — byte-identical to ORIGINAL (the calibration baseline).
- `dedup` — **one fewer message**: the duplicate `user` line is gone, but every
  tool-paired message survives. Lossless.
- `prune` — **same message count**, but the *oldest* tool outputs shrank to a
  head+tail with an elision marker. This is the lossy one — real savings, possible
  quality cost.

Now look at an actual truncation, old vs recent tool message:

In [4]:
pruned    = apply_transform(SAMPLE, "prune")
tool_idxs = [i for i, m in enumerate(SAMPLE["messages"]) if m.get("role") == "tool"]
old_i, recent_i = tool_idxs[0], tool_idxs[-1]   # prune keeps the last few intact

print(f"--- OLD tool msg [{old_i}] (older than keep_last -> TRUNCATED) ---")
print("before:", len(SAMPLE['messages'][old_i]['content']), "chars")
print("after :", len(pruned['messages'][old_i]['content']), "chars\n")
print(pruned['messages'][old_i]['content'])

print(f"\n--- RECENT tool msg [{recent_i}] (within keep_last -> UNTOUCHED) ---")
print("before:", len(SAMPLE['messages'][recent_i]['content']), "chars")
print("after :", len(pruned['messages'][recent_i]['content']), "chars")

--- OLD tool msg [4] (older than keep_last -> TRUNCATED) ---
before: 1137 chars
after : 633 chars

output of step 0
log line ....
log line ....
log line ....
log line ....
log line ....
log line ....
log line ....
log line ....
log line ....
log line ....
log line ....
log line ....
log line ....
log line ....
log line ....
log line ....
log line ....
log line ....
log line ....
log line ....
log line ....
log line ....
log line ....
log line ....
log line ....
log line ....
log line ....
log l
...[537 chars elided by rig]...
...
log line ....
log line ....
log line ....
log line ....
log line ....
log line ....
log line ....
log line ....
log line ....
log line ....
log line ....
log line ....
log line ....
log line ....


--- RECENT tool msg [14] (within keep_last -> UNTOUCHED) ---
before: 1137 chars
after : 1137 chars


**Experiments to try right here (no model needed):**
- `make_convo(n_turns=2)` — too few tool turns for `prune` to fire (keep_last=3). Why is that an honest result?
- `make_convo(dup=False)` — `dedup` now changes nothing. When does dedup actually help?
- `make_convo(output_size=5)` — outputs below the prune threshold stay whole.
- Call `transforms.prune_old_tool_outputs(copy.deepcopy(SAMPLE), keep_last=1, head=50, tail=20)` to crank the knobs.

In [5]:
# scratch cell — change params and re-run
demo = make_convo(n_turns=2, dup=False)
show(demo, "DEMO original")
show(apply_transform(demo, "dedup"), "DEMO dedup")
show(apply_transform(demo, "prune"), "DEMO prune")


=== DEMO original ===  7 msgs, ~2361 chars (~590 tok)
  [ 0] system        42 chars  
  [ 1] user          30 chars  
  [ 2] assistant      0 chars  tool-paired
  [ 3] tool        1137 chars  tool-paired
  [ 4] assistant      0 chars  tool-paired
  [ 5] tool        1137 chars  tool-paired
  [ 6] user          15 chars  

=== DEMO dedup ===  7 msgs, ~2361 chars (~590 tok)
  [ 0] system        42 chars  
  [ 1] user          30 chars  
  [ 2] assistant      0 chars  tool-paired
  [ 3] tool        1137 chars  tool-paired
  [ 4] assistant      0 chars  tool-paired
  [ 5] tool        1137 chars  tool-paired
  [ 6] user          15 chars  

=== DEMO prune ===  7 msgs, ~2361 chars (~590 tok)
  [ 0] system        42 chars  
  [ 1] user          30 chars  
  [ 2] assistant      0 chars  tool-paired
  [ 3] tool        1137 chars  tool-paired
  [ 4] assistant      0 chars  tool-paired
  [ 5] tool        1137 chars  tool-paired
  [ 6] user          15 chars  


## 3. Metering — the token/cost accounting

`proxy.py` reads OpenAI usage off the response: `prompt_tokens` -> **input**,
`completion_tokens` -> **output**. There is **no cache-read token** on local
models, which is why the rig's headroom signals are input-token growth and
savings rather than cache-hit ratio.

`meter.call_cost` is ~0 locally because `PRICES` is empty — populate it only if
you A/B against a paid endpoint.

In [6]:
fake_usage = {"input_tokens": 5321, "output_tokens": 128, "model": "llama3.1"}
print("PRICES registered :", meter.PRICES or "{}  -> local cost is 0")
print("call_cost         :", meter.call_cost(fake_usage), "USD")

# A logged row is exactly this shape (one JSONL line per call -> rig_calls.jsonl):
print("\nlog row shape:")
print(json.dumps({
    "ts": "<unix>", "run_id": "<proxy start>", "session": "<first-msg hash[:12]>",
    "transform": transforms.transform_name(), "n_messages": len(SAMPLE["messages"]),
    "usage": fake_usage, "cost_usd": meter.call_cost(fake_usage),
    "latency_s": 1.23, "text_len": 0, "text_head": "<first 200 chars of reply>",
}, indent=2))

PRICES registered : {}  -> local cost is 0
call_cost         : 0.0 USD

log row shape:
{
  "ts": "<unix>",
  "run_id": "<proxy start>",
  "session": "<first-msg hash[:12]>",
  "transform": "identity",
  "n_messages": 16,
  "usage": {
    "input_tokens": 5321,
    "output_tokens": 128,
    "model": "llama3.1"
  },
  "cost_usd": 0.0,
  "latency_s": 1.23,
  "text_len": 0,
  "text_head": "<first 200 chars of reply>"
}


## 4. Live round-trip (optional — needs the proxy or Ollama running)

The single most clarifying trick: **send the same request to the proxy and to
Ollama directly, and diff.** Everything is identical except what the proxy adds
(the transform, `stream_options.include_usage` injection, and metering).

Boot the proxy first in another terminal:
```bash
RIG_TRANSFORM=identity .venv/bin/uvicorn proxy:app --port 8787
```
If nothing is running, these cells just print a skip message.

In [7]:
import httpx
PROXY  = "http://localhost:8787/v1/chat/completions"
OLLAMA = "http://localhost:11434/v1/chat/completions"
HEADERS = {"authorization": "Bearer ollama"}   # Ollama ignores auth

def call(url, stream=False):
    body = {"model": "llama3.1", "stream": stream,
            "messages": [{"role": "user", "content": "say hi in exactly 3 words"}]}
    if stream:
        body["stream_options"] = {"include_usage": True}
    r = httpx.post(url, json=body, headers=HEADERS, timeout=120)
    r.raise_for_status()
    return r.json()

for label, url in [("PROXY", PROXY), ("OLLAMA direct", OLLAMA)]:
    try:
        data = call(url)
        text = "".join(c["message"]["content"] for c in data["choices"])
        print(f"{label:<13} text={text!r}  usage={data.get('usage')}")
    except Exception as e:
        print(f"{label:<13} skip ({type(e).__name__}) — endpoint not up?")

PROXY         skip (ConnectError) — endpoint not up?
OLLAMA direct skip (HTTPStatusError) — endpoint not up?


### Watch the raw SSE stream

This is the literal **output** wire format the proxy passes through unbuffered and
parses out-of-band. Note the `delta.content` chunks, then a terminal chunk
carrying `usage` (only because the proxy injected `include_usage`).

In [ ]:
def watch_stream(url, max_lines=12):
    body = {"model": "llama3.1", "stream": True,
            "stream_options": {"include_usage": True},
            "messages": [{"role": "user", "content": "count to five"}]}
    with httpx.stream("POST", url, json=body, headers=HEADERS, timeout=120) as resp:
        shown = 0
        for line in resp.iter_lines():
            if not line:
                continue
            print(line[:200])
            shown += 1
            if shown >= max_lines:
                print("... (truncated)")
                break

try:
    watch_stream(PROXY)
except Exception as e:
    print(f"skip ({type(e).__name__}) — start the proxy/Ollama to see the stream")

## 5. Capture & dissect REAL agent traffic

Synthetic requests are great for learning the shape; real ones show how fast the
tail actually balloons. Capture them by booting the proxy with `RIG_DUMP`:

```bash
RIG_DUMP=reqs.jsonl RIG_TRANSFORM=identity .venv/bin/uvicorn proxy:app --port 8787
# then, in another terminal, drive one small task:
export OPENAI_API_KEY=ollama OPENAI_API_BASE=http://localhost:8787/v1
.venv/bin/mini -m openai/llama3.1 -y -t "add a docstring to app.py"
```

Each *original* request body is appended to `reqs.jsonl`. Then run the cell below
to dissect them with the same `show()` helper.

In [ ]:
DUMP = "reqs.jsonl"
if os.path.exists(DUMP):
    reqs = [json.loads(l) for l in open(DUMP) if l.strip()]
    print(f"{len(reqs)} captured requests; inspecting the LAST (deepest context)\n")
    r = reqs[-1]
    show(r, "captured ORIGINAL")
    show(apply_transform(r, "dedup"), "captured + dedup")
    show(apply_transform(r, "prune"), "captured + prune")
else:
    print("no reqs.jsonl yet — see the boot command above.")

## 6. Read the logs

`rig_data` is the shared loader the dashboard and analysis notebook also use, so
these numbers can't drift from the official charts.

- `rig_calls.jsonl` — live proxy traffic (one row per call), with a derived `turn`
  index for context-growth plots.
- `ab_results.jsonl` — `ab_runner.py` divergence + token-savings.

In [ ]:
calls = rig_data.load_calls()
ab    = rig_data.load_ab()
print("rig_calls.jsonl :", calls.shape)
print("ab_results.jsonl:", ab.shape)
if not ab.empty:
    print("noise floor (identity-vs-identity divergence):", rig_data.noise_floor(ab))
if calls.empty:
    print("no proxy calls logged yet — run the proxy behind mini-swe-agent first")
else:
    display(calls.head())

---
### Cheat sheet — knobs & entry points

| env var | what it does | default |
|---|---|---|
| `RIG_TRANSFORM` | which transform the proxy applies | `identity` |
| `RIG_UPSTREAM_BASE` | OpenAI-compatible upstream | `http://localhost:11434/v1` |
| `RIG_PORT` | proxy listen port | `8787` |
| `RIG_DUMP` | append original requests to this JSONL | (off) |
| `RIG_LOG` | per-call meter log | `rig_calls.jsonl` |
| `RIG_AB_LOG` | ab_runner output | `ab_results.jsonl` |

- `GET /health` — current transform + upstream
- `python meter.py` — per-transform token totals table
- `python ab_runner.py reqs.jsonl <transform>` — divergence vs noise floor
- `streamlit run dashboard.py` — the three live charts